# Process Flow Check
Notebook is to check that the functions and general dataflow process is working as expected

In [1]:
import os
import sys
from dotenv import load_dotenv
import json
import pandas as pd
import numpy as np
from google.cloud import bigquery
import pydata_google_auth
import requests
import time
from datetime import timedelta, datetime, date
import urllib3
from pathlib import Path


# adding project directory for easier import
notebook_dir = Path(os.getcwd())
parent_dir = str(notebook_dir.parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)


from scripts.queries import *
from scripts.constants import SITE_MAPPINGS, SITE_BURN_DATATYPES, SITE_GENERATION_DATATYPES, AVAILABILITY_FILES_XLSX, AVAILABILITY_FILES_CSV
from scripts.data_pull_functions import run_date_parameterized_query, run_generation_query,  pull_unit_availability, pull_yes_forecast_historical, pull_yes_actual_historical
from scripts.data_clean_functions import clean_generation_unit_data, clean_yes_actual, clean_generation_unit_data, clean_yes_forecast

In [2]:
# Google Cloud Credentials
credentials = pydata_google_auth.get_user_credentials(
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
    auth_local_webserver=True,
)

client = bigquery.Client(project="bepc-prj-energy-prod", credentials=credentials )

In [3]:
# API Credentials
yes_username = os.getenv('YES_USERNAME')
yes_password = os.getenv('YES_PASSWORD')

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [4]:
# Dates for data pulls
start = '2026-06-01'
gas_start = '2026-05-31'
end = str(date.today() + pd.Timedelta(14, unit='D'))

print(f'start date: {start}     end date: {end}')

start date: 2026-06-01     end date: 2026-08-19


### Burn Data

In [5]:
gas_burn_daily_df = run_date_parameterized_query(client, GAS_BURN_QUERY, gas_start, end, SITE_BURN_DATATYPES)

gas_burn_daily_df['site'] = gas_burn_daily_df['marketarea'].replace(SITE_MAPPINGS)
gas_burn_daily_df = gas_burn_daily_df[['gas_day', 'site', 'energy']].rename(columns={'energy': 'gas_burn_MMBtu'})

print(gas_burn_daily_df.shape)

(405, 3)


### Generation Data

In [6]:
generation_df = run_generation_query(client, [PGS_GENERATION_QUERY, NON_PGS_GENERATION_QUERY], start, end, SITE_GENERATION_DATATYPES)
site_generation_df = clean_generation_unit_data(generation_df)['hourly_site_generation_df']
site_generation_by_gas_day_df = clean_generation_unit_data(generation_df)['daily_site_generation_by_gas_day_df']


### Availability Data

In [7]:
site_availability_df = pull_unit_availability(AVAILABILITY_FILES_XLSX, AVAILABILITY_FILES_CSV, start, end)['site_availability_df']

### Yes Energy Forecast Data

In [37]:
yes_historical_forecast_df = pull_yes_forecast_historical(yes_username, yes_password, start_date=start, end_date = end)
yes_historical_forecast_cleaned_df = clean_yes_forecast(yes_historical_forecast_df)


Pulling forecast: 2026-06-01 ---> 2026-06-30
Pulling forecast: 2026-07-01 ---> 2026-07-31
Pulling forecast: 2026-08-01 ---> 2026-08-19


### Yes Energy Actual Data

In [9]:
yes_historical_actual_df = pull_yes_actual_historical(yes_username, yes_password, start_date=start, end_date = end)
yes_historical_actual_cleaned_df = clean_yes_actual(yes_historical_actual_df)

Pulling actuals: 2026-06-01 ---> 2026-06-30
Pulling actuals: 2026-07-01 ---> 2026-07-31
Pulling actuals: 2026-08-01 ---> 2026-08-19


### Merging Datasets

Process starts with the hourly site generation

In [31]:
#datetime is hour ending
site_generation_df[(site_generation_df['site'] == 'DCS') & (site_generation_df['datetime'] >= '2026-07-31')& (site_generation_df['datetime'] <= '2026-08-01')]

,datetime,gas_day,hour,site,hourly_site_gen_mw
7196,2026-07-31 00:00:00,2026-07-30,he24,DCS,281.0
7201,2026-07-31 01:00:00,2026-07-30,he01,DCS,286.0
7206,2026-07-31 02:00:00,2026-07-30,he02,DCS,278.0
7211,2026-07-31 03:00:00,2026-07-30,he03,DCS,270.0
7216,2026-07-31 04:00:00,2026-07-30,he04,DCS,226.0
7221,2026-07-31 05:00:00,2026-07-30,he05,DCS,160.0
7226,2026-07-31 06:00:00,2026-07-30,he06,DCS,156.0
7231,2026-07-31 07:00:00,2026-07-30,he07,DCS,140.0
7236,2026-07-31 08:00:00,2026-07-30,he08,DCS,141.0
7241,2026-07-31 09:00:00,2026-07-30,he09,DCS,169.0


In [67]:
# adding daily burn information to the generation data based on gas day
# Some NaN in MMBtu are from gas day shift
print(site_generation_df.shape)
merged_df = site_generation_df.merge(gas_burn_daily_df, how='left', on=['gas_day', 'site'])

(9600, 5)


In [68]:
merged_df = merged_df.merge(site_generation_by_gas_day_df, how='left', on=['gas_day', 'site'])

# calculating hourly gas burn
# need to decide how to handle zeros in denominator
merged_df['hourly_gas_burn'] = round((merged_df['gas_burn_MMBtu'] / merged_df['daily_site_gen_mw']) * merged_df['hourly_site_gen_mw'], 2)
#merged_df.head(34)

In [69]:
# adding site availability to the merged dataframe
merged_df = merged_df.merge(site_availability_df, how='left', on=['datetime', 'site'])
merged_df['utilization'] = merged_df['hourly_site_gen_mw'] / merged_df['availability_mw']
#print(merged_df.shape)
#merged_df.head(10)

In [70]:
# adding yes historical forecast data
merged_df = merged_df.merge(yes_historical_forecast_cleaned_df, how='left', on=['datetime'])
#merged_df.head()

In [71]:
# adding yes historical actual data
merged_df = merged_df.merge(yes_historical_actual_cleaned_df, how='left', on=['datetime'])
print(merged_df.shape)
#merged_df.head()

(9600, 24)
